# Case study: clusters, maps and anomalies in the decisions of one chapter

*Session 11, blocks 2 and 3 practice. Author: course team, licence CC-BY-4.0.*

We take the decisions of **chapter 94** (furniture, seats, mattresses and bedding, lamps and light fittings) from the 50,000-decision sample and ask what structure they have without using the heading.

- **Part A (block 2):** k-means, hierarchical clustering and DBSCAN on two representations: the description (in many languages) and the English keywords. Evaluate with silhouette, stability and profiles; use the heading and the language only to *interpret* the clusters.
- **Part B (block 3):** a map with truncated SVD and t-SNE, decisions far from their heading's centroid (anomalies), and a test whether cluster features improve a gradient-boosting model for the heading on a time-based split.

Keywords and headings are allowed here because we explore; the heading model in Part B uses only the description. Theory: [page 2](../theory/02-hierarchical-dbscan-and-evaluation.md), [page 3](../theory/03-dimensionality-reduction.md), [page 4](../theory/04-anomaly-detection-and-unsupervised-features.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, adjusted_rand_score, f1_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

sample = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
furn = sample[sample["chapter"] == "94"].reset_index(drop=True)
print(len(furn))
furn["heading"].value_counts().to_frame().join(nomenclature["heading_description"].str[:60])

# Part A: three clustering methods, two representations

## A.1 From text to 50 numbers per decision

In [ ]:
def lsa(k=50):
    return make_pipeline(TruncatedSVD(k, random_state=0), Normalizer())


desc_vec = TfidfVectorizer(min_df=3, sublinear_tf=True, token_pattern=r"(?u)\b[^\W\d_]{2,}\b")
Z_desc = lsa().fit_transform(desc_vec.fit_transform(furn["description"]))
kw_vec = TfidfVectorizer(min_df=3, lowercase=False, token_pattern=None,
                         tokenizer=lambda s: [w.strip() for w in s.split(",") if w.strip()])
X_kw = kw_vec.fit_transform(furn["keywords"].fillna(""))
Z_kw = lsa().fit_transform(X_kw)
print(Z_desc.shape, Z_kw.shape)

## A.2 k-means for several k, on both representations

Silhouette (internal), and the agreement (ARI) with two variables that the clustering did not see: heading and language.

In [ ]:
rows = []
for name, Z in [("description", Z_desc), ("keywords", Z_kw)]:
    for k in (2, 3, 4, 5, 6, 8):
        labels = KMeans(k, n_init=10, random_state=0).fit_predict(Z)
        rows.append({"representation": name, "k": k, "silhouette": silhouette_score(Z, labels),
                     "ARI heading": adjusted_rand_score(furn["heading"], labels),
                     "ARI language": adjusted_rand_score(furn["language"], labels)})
pd.DataFrame(rows).set_index(["representation", "k"]).round(3)

**Question.** Which representation gives clusters of *products*, which gives clusters of *languages*? Which k has the highest silhouette for the descriptions, and what do those clusters represent?

## A.3 Hierarchical clustering (Ward) on the keyword components

In [ ]:
Z_ward = linkage(Z_kw, method="ward")
fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(Z_ward, truncate_mode="lastp", p=25, ax=ax)
ax.set_title("Ward dendrogram of the chapter-94 decisions (keyword components), top 25 merges")
ward4 = fcluster(Z_ward, t=4, criterion="maxclust")
km4 = KMeans(4, n_init=10, random_state=0).fit(Z_kw)
print("silhouette k-means", round(silhouette_score(Z_kw, km4.labels_), 3),
      "| Ward", round(silhouette_score(Z_kw, ward4), 3),
      "| ARI between them", round(adjusted_rand_score(km4.labels_, ward4), 2))

## A.4 DBSCAN

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z_kw).kneighbors(Z_kw)
print("10th-neighbour distance quantiles:", np.quantile(dist[:, -1], [0.5, 0.9, 0.95, 0.99]).round(2))
for eps in (0.3, 0.5, 0.7):
    labels = pd.Series(DBSCAN(eps=eps, min_samples=10).fit_predict(Z_kw))
    print(eps, labels.nunique() - (labels == -1).any(), "clusters,", (labels == -1).sum(), "noise")

## A.5 Stability and profile of the chosen clustering

In [ ]:
rng = np.random.default_rng(0)
aris = []
for seed in range(10):
    idx = rng.choice(len(Z_kw), len(Z_kw), replace=True)
    aris.append(adjusted_rand_score(km4.labels_, KMeans(4, n_init=10, random_state=seed).fit(Z_kw[idx]).predict(Z_kw)))
print("bootstrap ARI min / median:", np.round([min(aris), np.median(aris)], 2))
furn["cluster"] = km4.labels_
terms = kw_vec.get_feature_names_out()
profile = pd.crosstab(furn["cluster"], furn["heading"])
profile["size"] = profile.sum(axis=1)
profile["top keywords"] = [", ".join(terms[np.asarray(X_kw[km4.labels_ == c].mean(axis=0)).ravel().argsort()[::-1][:4]])
                           for c in range(4)]
profile

**Exercise A.** Write a one-paragraph comparison of the three methods for a customs analyst: which clustering would you show, how stable is it, and how would you describe each cluster in one sentence?

# Part B: maps, anomalies and cluster features

## B.1 A t-SNE map of the description components

In [ ]:
S = TruncatedSVD(50, random_state=0).fit_transform(desc_vec.transform(furn["description"]))
text_map = TSNE(perplexity=30, init="pca", random_state=0).fit_transform(S)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col in zip(axes, ["language", "heading"]):
    for value in furn[col].value_counts().index[:5]:
        m = (furn[col] == value).to_numpy()
        ax.scatter(*text_map[m].T, s=3, label=value)
    ax.set_title(f"coloured by {col}")
    ax.legend(markerscale=3)
plt.tight_layout()

## B.2 Decisions far from their heading's centroid (German decisions only)

In [ ]:
de = furn[furn["language"] == "de"].reset_index(drop=True)
Z_de = make_pipeline(TfidfVectorizer(min_df=3, sublinear_tf=True), lsa()).fit_transform(de["description"])
centroids = pd.DataFrame(Z_de).groupby(de["heading"]).mean()
own = centroids.loc[de["heading"]].to_numpy()
de["similarity"] = (Z_de * own).sum(axis=1) / np.linalg.norm(own, axis=1)
unit = centroids.to_numpy() / np.linalg.norm(centroids.to_numpy(), axis=1, keepdims=True)
de["nearest heading"] = centroids.index[(Z_de @ unit.T).argmax(axis=1)]
print("share of decisions closer to another heading's centroid:", round((de["nearest heading"] != de["heading"]).mean(), 3))
de.nsmallest(10, "similarity")[["heading", "nearest heading", "similarity", "keywords"]].round(2)

**Exercise B.2.** Read the ten least typical decisions through their keywords and the heading texts (`nomenclature.loc["9405", "heading_description"]`). For each, decide: unusual product, borderline case, or possibly wrong heading? What would a customs specialist need to check?

## B.3 Do cluster features help a tree model?

Task: predict the heading of the chapter-94 decisions from the description only. Fit on 2017–2021, validate on 2022–2023. The vectoriser, the SVD and k-means are fitted on the fitting rows only. Features: 50 description components, with and without the distances to 8 k-means centroids.

In [ ]:
fit = (furn["start_date"] < "2022-01-01").to_numpy()
val = ~fit
y = furn["heading"].to_numpy()
vec = TfidfVectorizer(min_df=3, sublinear_tf=True)
reduce = lsa()
C_fit = reduce.fit_transform(vec.fit_transform(furn.loc[fit, "description"]))
C_val = reduce.transform(vec.transform(furn.loc[val, "description"]))
km8 = KMeans(8, n_init=10, random_state=0).fit(C_fit)
variants = {"50 components": (C_fit, C_val),
            "50 components + 8 centroid distances": (np.hstack([C_fit, km8.transform(C_fit)]),
                                                     np.hstack([C_val, km8.transform(C_val)]))}
rows = []
for name, (A, B) in variants.items():
    model = HistGradientBoostingClassifier(l2_regularization=1.0, random_state=0).fit(A, y[fit])
    p = model.predict(B)
    rows.append({"features": name, "accuracy": accuracy_score(y[val], p),
                 "macro-F1": f1_score(y[val], p, average="macro", zero_division=0)})
print(fit.sum(), "fitting rows,", val.sum(), "validation rows")
pd.DataFrame(rows).set_index("features").round(3)

**Questions.**

1. Do the centroid distances help? Is the difference larger than what you would expect from noise with this many validation rows (compute a bootstrap interval as in Session 10)?
2. Why must k-means be fitted on the fitting rows only?

## Exercises

1. Repeat Part A for chapter 64 (footwear) or 39 (plastics). Do the description clusters again follow the languages?
2. Cluster only the German descriptions of chapter 94. Does the agreement with the heading improve compared with all languages?
3. Replace the 8 centroid distances in B.3 by the one-hot cluster label. Which works better, and why might distances be smoother features?